# 🧠 GodelAI-Lite: Enhancing Gemma 4 with Memory & Identity Continuity
### Gemma 4 Good Hackathon Submission

**Core Idea:** Small models + Structured Memory = Cumulative Intelligence

## 1. Setup & Installation

In [ ]:
!pip install transformers accelerate torch huggingface_hub

In [ ]:
import torch
import json
from typing import Dict, List, Optional
from dataclasses import dataclass, field, asdict
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import warnings
warnings.filterwarnings('ignore')

## 2. MemPalace-Lite: Structured Memory Layer

In [ ]:
@dataclass
class MemoryEntry:
    """Single memory entry"""
    content: str
    timestamp: int
    relevance_score: float = 1.0
    category: str = "general"

class MemPalaceLite:
    """
    Structured external memory for SLMs
    Stores: history, key facts, reasoning patterns
    """
    
    def __init__(self, max_history: int = 10, max_facts: int = 20):
        self.history: List[MemoryEntry] = []
        self.key_facts: List[MemoryEntry] = []
        self.patterns: List[str] = []
        self.max_history = max_history
        self.max_facts = max_facts
        self.step_counter = 0
    
    def add_to_history(self, interaction: str, category: str = "interaction"):
        """Add interaction to episodic memory"""
        entry = MemoryEntry(
            content=interaction,
            timestamp=self.step_counter,
            category=category
        )
        self.history.append(entry)
        # Keep only recent history
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
    
    def add_fact(self, fact: str, relevance: float = 1.0):
        """Extract and store key facts"""
        entry = MemoryEntry(
            content=fact,
            timestamp=self.step_counter,
            relevance_score=relevance,
            category="fact"
        )
        self.key_facts.append(entry)
        # Keep most relevant facts
        if len(self.key_facts) > self.max_facts:
            self.key_facts.sort(key=lambda x: x.relevance_score, reverse=True)
            self.key_facts = self.key_facts[:self.max_facts]
    
    def add_pattern(self, pattern: str):
        """Store useful reasoning patterns"""
        if pattern not in self.patterns:
            self.patterns.append(pattern)
    
    def get_context(self) -> str:
        """Build context string from memory"""
        context_parts = []
        
        # Add key facts
        if self.key_facts:
            facts_text = "\n".join([f"• {f.content}" for f in self.key_facts[-5:]])
            context_parts.append(f"[KEY FACTS]\n{facts_text}")
        
        # Add recent history
        if self.history:
            history_text = "\n".join([f"[{h.timestamp}] {h.content}" for h in self.history[-3:]])
            context_parts.append(f"[RECENT HISTORY]\n{history_text}")
        
        # Add patterns
        if self.patterns:
            patterns_text = "\n".join([f"→ {p}" for p in self.patterns[-2:]])
            context_parts.append(f"[REASONING PATTERNS]\n{patterns_text}")
        
        return "\n\n".join(context_parts)
    
    def extract_facts(self, text: str) -> List[str]:
        """Simple fact extraction (can be enhanced with NER)"""
        # Heuristic: extract sentences with specific patterns
        facts = []
        sentences = text.replace('.', '\n').split('\n')
        for sent in sentences:
            sent = sent.strip()
            # Extract factual statements
            if any(kw in sent.lower() for kw in ['is', 'are', 'has', 'contains', 'defined']):
                if len(sent) > 10 and len(sent) < 200:
                    facts.append(sent)
        return facts[:3]  # Top 3 facts
    
    def to_dict(self) -> Dict:
        return {
            "history": [asdict(h) for h in self.history],
            "key_facts": [asdict(f) for f in self.key_facts],
            "patterns": self.patterns
        }

## 3. MACP-Lite: Continuity Layer

In [ ]:
@dataclass
class ReasoningStep:
    """Single step in reasoning chain"""
    step_id: int
    input_context: str
    model_output: str
    confidence: float
    next_action: str

class MACPLite:
    """
    Multi-Agent Continuity Protocol (Lite)
    Ensures consistent context passing and structured reasoning flow
    """
    
    def __init__(self):
        self.reasoning_chain: List[ReasoningStep] = []
        self.current_step = 0
        self.context_buffer: str = ""
    
    def start_chain(self, initial_input: str):
        """Initialize reasoning chain"""
        self.reasoning_chain = []
        self.current_step = 0
        self.context_buffer = initial_input
    
    def add_step(self, input_ctx: str, output: str, confidence: float, next_action: str):
        """Add step to reasoning chain"""
        step = ReasoningStep(
            step_id=self.current_step,
            input_context=input_ctx,
            model_output=output,
            confidence=confidence,
            next_action=next_action
        )
        self.reasoning_chain.append(step)
        self.current_step += 1
        self.context_buffer = output
    
    def get_chain_summary(self) -> str:
        """Summarize the reasoning chain"""
        if not self.reasoning_chain:
            return "No reasoning chain yet."
        
        summary = ["[REASONING CHAIN]"]
        for step in self.reasoning_chain[-5:]:  # Last 5 steps
            summary.append(f"Step {step.step_id}: {step.model_output[:100]}...")
        return "\n".join(summary)
    
    def should_continue(self, max_steps: int = 5) -> bool:
        """Check if reasoning should continue"""
        return self.current_step < max_steps

## 4. GIFP-Lite: Identity Layer

In [ ]:
class GIFPLite:
    """
    Goal-Oriented Identity & Fixed Protocol (Lite)
    Maintains consistent agent identity and behavior
    """
    
    def __init__(self, role: str = "helpful assistant"):
        self.role_definition = role
        self.constraints: List[str] = []
        self.behavior_history: List[str] = []
        self.drift_threshold = 0.3  # Allow some flexibility
    
    def set_constraints(self, constraints: List[str]):
        """Set behavioral constraints"""
        self.constraints = constraints
    
    def get_identity_prompt(self) -> str:
        """Generate identity system prompt"""
        prompt = f"You are a {self.role_definition}.\n"
        
        if self.constraints:
            prompt += "\nConstraints:\n"
            for c in self.constraints:
                prompt += f"- {c}\n"
        
        prompt += "\nMaintain consistency across all interactions.\n"
        return prompt
    
    def check_consistency(self, output: str) -> float:
        """
        Simple consistency check
        Returns score 0-1 (1 = fully consistent)
        """
        # Basic heuristic: check for contradiction patterns
        contradiction_words = ['actually', 'never mind', 'scratch that', 'i was wrong']
        output_lower = output.lower()
        
        contradictions = sum(1 for word in contradiction_words if word in output_lower)
        
        # Simple score
        score = max(0, 1 - (contradictions * 0.2))
        return score
    
    def record_behavior(self, output: str):
        """Track behavior for drift analysis"""
        self.behavior_history.append(output[:100])
        if len(self.behavior_history) > 20:
            self.behavior_history = self.behavior_history[-20:]

## 5. GodelAI-Lite: Main System

In [ ]:
class GodelAILite:
    """
    Main GodelAI-Lite System
    Combines Gemma 4 with MemPalace-Lite, MACP-Lite, and GIFP-Lite
    """
    
    def __init__(self, model_name: str = "google/gemma-2b"):
        print(f"Loading model: {model_name}...")
        
        # Load model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        
        # Initialize components
        self.memory = MemPalaceLite()
        self.continuity = MACPLite()
        self.identity = GIFPLite(role="helpful AI assistant with persistent memory")
        
        # Set identity constraints
        self.identity.set_constraints([
            "Always be helpful and accurate",
            "Reference previous information when relevant",
            "Maintain logical consistency",
            "Acknowledge uncertainty when present"
        ])
        
        print("GodelAI-Lite initialized! 🚀")
    
    def build_prompt(self, user_input: str) -> str:
        """Build augmented prompt with memory context"""
        
        # Get identity prompt
        identity_prompt = self.identity.get_identity_prompt()
        
        # Get memory context
        memory_context = self.memory.get_context()
        
        # Build full prompt
        prompt = f"""{identity_prompt}

{memory_context}

[CURRENT INPUT]
User: {user_input}

Assistant:"""
        
        return prompt
    
    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        """Generate response from model"""
        
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        gen_config = GenerationConfig(
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )
        
        outputs = self.model.generate(**inputs, generation_config=gen_config)
        
        # Decode only the new tokens
        input_length = inputs.input_ids.shape[1]
        generated = outputs[0, input_length:]
        response = self.tokenizer.decode(generated, skip_special_tokens=True)
        
        return response.strip()
    
    def chat(self, user_input: str, refine: bool = True, max_refinements: int = 2) -> str:
        """
        Main chat interface with memory and refinement
        """
        
        # Start reasoning chain
        self.continuity.start_chain(user_input)
        
        # Build prompt with memory
        prompt = self.build_prompt(user_input)
        
        # Generate response
        response = self.generate(prompt)
        
        # Check consistency
        consistency_score = self.identity.check_consistency(response)
        
        # Add to continuity chain
        self.continuity.add_step(
            input_ctx=user_input,
            output=response,
            confidence=consistency_score,
            next_action="continue" if consistency_score > 0.7 else "refine"
        )
        
        # Extract and store facts
        new_facts = self.memory.extract_facts(response)
        for fact in new_facts:
            self.memory.add_fact(fact)
        
        # Store interaction
        self.memory.add_to_history(f"User: {user_input}", "user_input")
        self.memory.add_to_history(f"Assistant: {response}", "assistant_output")
        
        # Record behavior
        self.identity.record_behavior(response)
        
        # Iterative refinement (optional)
        if refine and consistency_score < 0.8 and max_refinements > 0:
            refinement_prompt = f"Let me refine that response for better clarity and accuracy."
            self.memory.add_to_history(refinement_prompt, "refinement")
            refined_response = self.chat(f"Please refine: {response}", refine=False)
            return f"{response}\n\n[Refined]: {refined_response}"
        
        return response
    
    def get_memory_state(self) -> Dict:
        """Return current memory state"""
        return self.memory.to_dict()
    
    def get_reasoning_chain(self) -> str:
        """Return reasoning chain summary"""
        return self.continuity.get_chain_summary()

## 6. Demo & Testing

In [ ]:
# Initialize GodelAI-Lite
# Using gemma-2b for faster loading (upgrade to gemma-7b if GPU allows)

assistant = GodelAILite(model_name="google/gemma-2b")

In [ ]:
# Test 1: Basic conversation with memory
print("=" * 50)
print("TEST 1: Multi-turn conversation with memory")
print("=" * 50)

response1 = assistant.chat("My name is Alex and I'm a software engineer.")
print(f"\nTurn 1: {response1}")

response2 = assistant.chat("What do you know about me?")
print(f"\nTurn 2: {response2}")

response3 = assistant.chat("Can you help me write a Python function?")
print(f"\nTurn 3: {response3}")

In [ ]:
# Test 2: Check memory state
print("\n" + "=" * 50)
print("MEMORY STATE")
print("=" * 50)

memory_state = assistant.get_memory_state()
print(f"History entries: {len(memory_state['history'])}")
print(f"Key facts: {len(memory_state['key_facts'])}")
print(f"Patterns: {len(memory_state['patterns'])}")

print("\nKey Facts Stored:")
for fact in memory_state['key_facts']:
    print(f"  • {fact['content']}")

In [ ]:
# Test 3: Reasoning chain
print("\n" + "=" * 50)
print("REASONING CHAIN")
print("=" * 50)

print(assistant.get_reasoning_chain())

In [ ]:
# Test 4: Complex reasoning task
print("\n" + "=" * 50)
print("TEST 4: Complex reasoning with iterative refinement")
print("=" * 50)

response = assistant.chat(
    "If I have 3 apples and buy 5 more, then give away 2, how many do I have? "
    "Explain your reasoning step by step.",
    refine=True
)
print(response)

## 7. Evaluation Metrics

In [ ]:
# Simple evaluation framework
from typing import Tuple

def evaluate_memory_retention(assistant: GodelAILite) -> Tuple[float, str]:
    """
    Evaluate memory retention across conversation turns
    """
    # Reset
    assistant.memory = MemPalaceLite()
    
    # Inject fact
    test_fact = "The capital of France is Paris."
    assistant.chat(test_fact)
    
    # Query after several turns
    assistant.chat("What's the weather like?")
    assistant.chat("Tell me a joke.")
    response = assistant.chat("What did I tell you earlier?")
    
    # Check if fact was retained
    retained = "paris" in response.lower() or "france" in response.lower()
    
    return (1.0 if retained else 0.0, response)

def evaluate_consistency(assistant: GodelAILite) -> float:
    """
    Evaluate response consistency across multiple queries
    """
    scores = []
    
    for i in range(5):
        response = assistant.chat(f"Question {i}: What is 2+2?")
        score = assistant.identity.check_consistency(response)
        scores.append(score)
    
    return sum(scores) / len(scores)

# Run evaluations
print("Running evaluations...\n")

memory_score, memory_response = evaluate_memory_retention(assistant)
print(f"Memory Retention Score: {memory_score:.2f}")
print(f"Sample response: {memory_response[:100]}...\n")

consistency_score = evaluate_consistency(assistant)
print(f"Consistency Score: {consistency_score:.2f}")

## 8. Results Summary

In [ ]:
print("\n" + "=" * 50)
print("GODELAI-LITE: RESULTS SUMMARY")
print("=" * 50)

print(f"""
Architecture Components:
  ✓ MemPalace-Lite (Memory Layer)
  ✓ MACP-Lite (Continuity Layer)
  ✓ GIFP-Lite (Identity Layer)

Base Model: Gemma 4 (2B/7B)

Key Features:
  • Episodic memory for conversation history
  • Fact extraction and retention
  • Reasoning pattern storage
  • Identity consistency checking
  • Iterative refinement loop

Advantages:
  • No additional training required
  • Works with any SLM
  • Computationally lightweight
  • Cumulative intelligence over time

Conclusion: Small models + Memory = Enhanced Intelligence
```

## 9. References

- [MemPalace](https://github.com/milla-jovovich/mempalace) – Inspiration for structured memory systems
- [Gemma Models](https://huggingface.co/google) – Google's Gemma model family
- [Zenodo: GodelAI Framework](https://zenodo.org/records/18048374) – Full framework publication

---
**Gemma 4 Good Hackathon Submission**

*"Intelligence can scale through memory, not just parameters."*